In [23]:
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score,
    GridSearchCV, RandomizedSearchCV
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

print("scikit-learn:", sklearn.__version__)

scikit-learn: 1.6.1


In [24]:
df=pd.read_csv("/content/Breast Cancer Wisconsin (Diagnostic) Data Set.csv")

In [25]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [27]:
np.unique(df["diagnosis"])

array(['B', 'M'], dtype=object)

In [28]:
df['diagnosis'].value_counts()

,count
diagnosis,
B,357
M,212


In [29]:
df.isnull().sum()

,0
id,0
diagnosis,0
radius_mean,0
texture_mean,0
perimeter_mean,0
area_mean,0
smoothness_mean,0
compactness_mean,0
concavity_mean,0
concave points_mean,0


In [30]:
df = df.drop(columns=["id", "Unnamed: 32"], errors="ignore")

assert "diagnosis" in df.columns, "CSV must contain diagnosis."
assert set(df["diagnosis"].dropna().unique()) == {"B", "M"}
assert df["diagnosis"].notna().all(), "Diagnosis must not be missing."

le = LabelEncoder()
df["diagnosis"] = le.fit_transform(df["diagnosis"])
print("Encoding:", dict(zip(le.classes_, le.transform(le.classes_))))
display(df["diagnosis"].value_counts())

Encoding: {'B': np.int64(0), 'M': np.int64(1)}


,count
diagnosis,
0,357
1,212


In [31]:
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]
assert all(pd.api.types.is_numeric_dtype(X[c]) for c in X.columns)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (455, 30)
X_test: (114, 30)
y_train: (455,)
y_test: (114,)


In [32]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

tree_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", DecisionTreeClassifier(random_state=42))
])

baseline_scores = cross_val_score(
    tree_model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1
)
print("Fold accuracies:", np.round(baseline_scores, 4))
print("Baseline CV accuracy:", round(baseline_scores.mean(), 4))
print("CV standard deviation:", round(baseline_scores.std(), 4))

Fold accuracies: [0.8901 0.967  0.8901 0.9121 0.956 ]
Baseline CV accuracy: 0.9231
CV standard deviation: 0.0326


In [33]:
param_grid = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [3, 5, None],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

grid_search = GridSearchCV(
    tree_model, param_grid, cv=cv, scoring="accuracy", n_jobs=1
)
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", round(grid_search.best_score_, 4))

Best parameters: {'model__criterion': 'entropy', 'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10}
Best CV accuracy: 0.9363


In [34]:
param_dist = {
    "model__criterion": ["gini", "entropy"],
    "model__max_depth": [None, 2, 3, 4, 5, 6, 8, 10],
    "model__min_samples_split": [2, 5, 10, 15, 20],
    "model__min_samples_leaf": [1, 2, 3, 4, 5],
    "model__max_features": [None, "sqrt", "log2"]
}

random_search = RandomizedSearchCV(
    tree_model, param_dist, n_iter=20, cv=cv, scoring="accuracy",
    random_state=42, n_jobs=1
)
random_search.fit(X_train, y_train)

print("Best parameters:", random_search.best_params_)
print("Best CV accuracy:", round(random_search.best_score_, 4))

Best parameters: {'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_features': 'log2', 'model__max_depth': 8, 'model__criterion': 'entropy'}
Best CV accuracy: 0.9385


In [35]:
forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(
        n_estimators=200, max_features="sqrt", random_state=42, n_jobs=1
    ))
])

forest_scores = cross_val_score(
    forest_model, X_train, y_train, cv=cv, scoring="accuracy", n_jobs=1
)
forest_model.fit(X_train, y_train)
print("Random Forest CV accuracy:", round(forest_scores.mean(), 4))

Random Forest CV accuracy: 0.9626


In [36]:
tree_model.fit(X_train, y_train)

baseline_pred = tree_model.predict(X_test)
grid_pred = grid_search.best_estimator_.predict(X_test)
random_pred = random_search.best_estimator_.predict(X_test)
forest_pred = forest_model.predict(X_test)

baseline_test = accuracy_score(y_test, baseline_pred)
grid_test = accuracy_score(y_test, grid_pred)
random_test = accuracy_score(y_test, random_pred)
forest_test = accuracy_score(y_test, forest_pred)

In [37]:
results = pd.DataFrame({
    "Model": ["Decision Tree baseline", "Decision Tree GridSearchCV",
              "Decision Tree RandomizedSearchCV", "Random Forest"],
    "CV Accuracy": [baseline_scores.mean(), grid_search.best_score_,
                    random_search.best_score_, forest_scores.mean()],
    "Test Accuracy": [baseline_test, grid_test, random_test, forest_test],
    "Best Params": [
        "Not tuned: default Decision Tree",
        grid_search.best_params_,
        random_search.best_params_,
        "Not tuned: n_estimators=200, max_features='sqrt'"
    ]
})

with pd.option_context("display.max_colwidth", None):
    display(results.style.format({"CV Accuracy": "{:.4f}",
                                  "Test Accuracy": "{:.4f}"}))

,Model,CV Accuracy,Test Accuracy,Best Params
0,Decision Tree baseline,0.9231,0.9298,Not tuned: default Decision Tree
1,Decision Tree GridSearchCV,0.9363,0.9649,"{'model__criterion': 'entropy', 'model__max_depth': 5, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10}"
2,Decision Tree RandomizedSearchCV,0.9385,0.9561,"{'model__min_samples_split': 2, 'model__min_samples_leaf': 5, 'model__max_features': 'log2', 'model__max_depth': 8, 'model__criterion': 'entropy'}"
3,Random Forest,0.9626,0.9649,"Not tuned: n_estimators=200, max_features='sqrt'"
